# BigQuery

Você deve estar se perguntando: por quê preciso aprender a usar BigQuery? E a resposta é muito simples. Imagine que você tem centenas de gigabytes, até mesmo alguns petabytes de dados e você precisa utilizá-los para treinar um modelo de inteligência artifical. A minha máquina e a sua máquina muito dificilmente seriam capazes de realizar essa tarefa, não acha? Este é o problema, agora vamos para a solução.

O Google Cloud possui um serviço, o BigQuery. O BigQuery é um data warehouse que abriga uma quantidade incrível de dados. Através de ferramentas que vamos utilizar nesse guia, podemos lidar com os dados sem precisar transferí-los para nossa máquina própria antes, eliminando completamente o problema anterior. Existem duas formas principais de fazer isso. Usando código SQL ou BigFrames. Primeiro, vamos te mostrar como consumir os dados utilizando SQL.

Antes de continuar, lembre-se de que o motivo de usar BigQuery se resume a conseguir lidar com uma quantidade absurda de dados sem precisar ter uma máquina capaz de manipulá-los e ao mesmo tempo não desperdiçar recursos do serviço do BigQuery desnecessariamente. Ou seja: temos de puxar do servidor unicamente aquilo que for absolutamente necessário. Afinal, um projeto de graça no BigQuery tem um limite de 1TB de requisições e no mundo real o limite custa dinheiro da empresa.

## Conectando ao Projeto no BigQuery

Quando criamos um projeto no BigQuery, duas coisas que precisamos fazer na interface do website são: baixar uma chave de acesso; copiar o id do projeto. Essas duas informações vão nos permitir conectar ao projeto no BigQuery. Para configurar a conexão, é necessário definir uma variável de ambiente com o nome "GOOGLE_APPLICATION_CREDENTIALS". Essa variável de ambiente deve receber o caminho até o arquivo JSON que a interface nos entregou ao baixar a chave.

Após definir a variável de ambiente, podemos criar um objeto de conexão da classe bigquery.Client. Essa classe receberá um argumento project que define o ID do projeto no BigQuery. De agora em diante, considere a conexão realizada. Para verificar, podemos executar um código SQL simples. Utilizamos o método query do bigquery.Client para executar nosso código SQL. O retorno desse método retorna um objeto que tem o método to_dataframe. Esse método vai transformar o resultado do método query (os dados retornados pela consulta SQL) em um dataframe do pandas.

In [1]:
import os
from reading_sql.config import GOOGLE_APPLICATION_CREDENTIALS_PATH
from reading_sql.config import PROJECT_ID
from google.cloud import bigquery

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(GOOGLE_APPLICATION_CREDENTIALS_PATH)

bigquery_client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT 1;
"""

dataframe = bigquery_client.query(query).to_dataframe()

dataframe.head()


,f0_
0,1


## Reduzindo Custos

O nível grátis do BigQuery é limitado, então a coisa mais importante a se fazer antes de continuar é reduzir drasticamente o que receberemos de volta com a consulta. Para fazer isso, é muito simples. Basta utilizar as palavras-chave TABLESAMPLE e SYSTEM do SQL logo após a definição do FROM. O parâmetro passado é uma porcentagem dos dados em que queremos fazer a consulta. Abaixo, estamos consultando apenas 0.00001% de todos os dados. Isso reduz e muito os custos, afinal, temos recursos limitados.

Um outro ponto que é crucial para lidar com a limitação de recursos é selecionar única e absolutamente as colunas que de fato precisamos para prosseguir com a exploração dos dados e treinamento de um modelo.

In [2]:
query = """
SELECT 
    id as user_id, 
    age, 
    gender, 
    country, 
    traffic_source, 
    created_at
FROM `bigquery-public-data.thelook_ecommerce.users`
TABLESAMPLE SYSTEM (0.00001 PERCENT);
"""

dataframe = bigquery_client.query(query).to_dataframe()

print(dataframe.head(n=100))

    user_id  age gender        country traffic_source  \
0      7043   12      M         Brasil         Search   
1     34134   12      F          Japan         Search   
2     75172   12      M          Japan         Search   
3     25946   12      M          Japan         Search   
4     37507   12      F  United States         Search   
..      ...  ...    ...            ...            ...   
95    30000   12      F         Brasil         Search   
96    65786   12      F         Brasil         Search   
97    60816   12      F         Brasil         Search   
98    37164   12      F         Brasil         Search   
99     8515   12      M         Brasil        Organic   

                         created_at  
0         2019-09-27 12:12:00+00:00  
1         2022-12-27 17:06:00+00:00  
2         2021-10-12 06:54:00+00:00  
3         2025-04-26 18:43:00+00:00  
4         2022-08-23 08:01:00+00:00  
..                              ...  
95        2019-06-10 13:38:00+00:00  
96        2

Uma outra ótima prática é rodar a query em dry run. Quando fazemos isso, conseguimos descobrir quanto espaço de fato os dados ocupam e podemos utilizar isso como parâmetro ao fazer consultas. Para executar uma query em dry run, basta criamos um objeto da classe bigquery.QueryJobConfig. Essa classe pode receber um argumento dry_run. Quando o dry_run receber o valor True, a query que utilizar esse objeto de configurações retornará apenas parâmetros da consulta sem dados concretos. Para executar a query em dry run, utilizamos o mesmo método query do cliente do bigquery, mas agora, além de passar a string com código SQL, também passamos um parâmetro job_config que vai receber o objeto de configurações. O resultado dessa consulta terá uma propriedade total_bytes_processed que retorna o tamanho da porção de dados que a consulta retornaria. É muito importante executar essa verificação antes de rodar consultas que podem ter um resultado muito pesado.

In [3]:
job_config = bigquery.QueryJobConfig(dry_run=True)

job = bigquery_client.query(query, job_config=job_config)

print(job.total_bytes_processed / (1024 ** 2))

4.308048248291016


### Salvando uma Snapshot dos Dados

Quando estivermos trabalhando no mundo real, não é adequado repetirmos a mesma requisição para obter exatamente os mesmos dados do BigQuery. O mais correto é salvar localmente uma snapshot dos dados que de fato vamos utilizar. Para isso, salvaremos um arquivo do tipo parquet que é um formato compactado dos dados. Nesse formato, cada coluna é gravada isoladamente e se pedirmos apenas uma determinada coluna, o leitor pula fisicamente no disco todas as outras colunas e lê apenas a que você pediu.

Para salvar um arquivo do tipo parquet, basta utilizar o método to_parquet que os dataframes do pandas têm. Como argumento, o método deve receber o caminho onde o arquivo deve ser salvo.

In [4]:
dataframe.to_parquet("../snapshots/users.parquet")

### Lendo Arquivos Parquet

Para ler um arquivo do tipo parquet, basta utilizar o método read_parquet do pandas, passando como argumento o caminho até o arquivo. Esse método retorna um dataframe que podemos usar exatamente da mesma forma que usamos quando importamos CSV, JSON ou qualquer outro formato.

In [5]:
import pandas

dataframe = pandas.read_parquet("../snapshots/users.parquet")

dataframe.head()

,user_id,age,gender,country,traffic_source,created_at
0,7043,12,M,Brasil,Search,2019-09-27 12:12:00+00:00
1,34134,12,F,Japan,Search,2022-12-27 17:06:00+00:00
2,75172,12,M,Japan,Search,2021-10-12 06:54:00+00:00
3,25946,12,M,Japan,Search,2025-04-26 18:43:00+00:00
4,37507,12,F,United States,Search,2022-08-23 08:01:00+00:00


## Parameterização

Ao utilizar parâmetros para filtrar os resultados da consulta, devemos sempre evitar utilizar f strings por 4 motivos:

1. Formatação de strings
2. Filtro com listas
3. Tratamento de tipos
4. SQL Injection

No exemplo abaixo, temos uma consulta que filtra os usuários da base de dados pelo país, gênero e idade. No código SQL da consulta, referenciamos os parâmetros usando @nome_do_parametro. Para configurar os nomes dos parâmetros, tipos e valores, passamos um parâmetro query_parameters para o bigquery.QueryJobConfig. Esse parâmetro recebe uma lista com objetos do tipo bigquery.ScalarQueryParameter. Essa classe recebe três argumentos: o nome do parâmetro, o tipo e o valor. Observe o exemplo e associe os tipos que o bigquery pede para cada valor. Para rodar a consulta utilizando os parâmetros, basta utilizar o método query do client passando como argumento o job_config.

In [6]:
query = """
SELECT
    id as user_id,
    age,
    gender, 
    country, 
    traffic_source, 
    created_at
FROM `bigquery-public-data.thelook_ecommerce.users`
TABLESAMPLE SYSTEM (0.00001 PERCENT)
WHERE country = @country
AND gender = @gender
AND age >= @age;
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("age", "INT64", 18),
        bigquery.ScalarQueryParameter("gender", "STRING", "M"),
        bigquery.ScalarQueryParameter("country", "STRING", "Brasil"),
    ]
)

dataframe = bigquery_client.query(query, job_config=job_config).to_dataframe()

dataframe.head()

,user_id,age,gender,country,traffic_source,created_at
0,43518,20,M,Brasil,Search,2021-07-31 05:16:00+00:00
1,94151,39,M,Brasil,Search,2025-03-22 14:22:00+00:00
2,36355,28,M,Brasil,Search,2021-07-28 01:59:00+00:00
3,11343,18,M,Brasil,Search,2019-02-09 08:25:00+00:00
4,50774,47,M,Brasil,Facebook,2022-06-30 06:55:00+00:00


### Exemplo com Lista

Para filtrar usando listas, podemos utilizar a classe bigquery.ArrayQueryParameter ao declarar os parâmetros da consulta no job_config. O primeiro argumento para a classe naturalmente será o nome do parâmetro, o segundo será o tipo de dado que a lista guarda e o terceiro, é claro, a lista. Ao referenciar na query usando o IN, devemos utilizar a função UNNEST encapsulando a referência.

In [7]:
query = """
SELECT 
    id AS user_id,
    age,
    gender,
    country, 
    traffic_source,
    created_at
FROM `bigquery-public-data.thelook_ecommerce.users`
TABLESAMPLE SYSTEM (0.00001 PERCENT)
WHERE age >= @age
AND gender = @gender
AND country IN UNNEST(@countries)
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("age", "INT64", 18),
        bigquery.ScalarQueryParameter("gender", "STRING", "M"),
        bigquery.ArrayQueryParameter("countries", "STRING", ["Brasil", "Japan", "United States"])
    ]
)

dataframe = bigquery_client.query(query, job_config=job_config).to_dataframe()

dataframe.head(n=10)

,user_id,age,gender,country,traffic_source,created_at
0,11343,18,M,Brasil,Search,2019-02-09 08:25:00+00:00
1,81014,18,M,Japan,Organic,2019-04-26 17:21:00+00:00
2,23397,18,M,United States,Search,2021-09-28 10:09:00+00:00
3,89208,18,M,Brasil,Search,2022-09-13 16:55:00+00:00
4,17795,18,M,United States,Organic,2021-11-04 16:06:00+00:00
5,41314,18,M,Brasil,Search,2026-09-01 04:45:00+00:00
6,28914,18,M,Brasil,Search,2024-08-02 13:32:00+00:00
7,16038,18,M,Brasil,Email,2019-03-08 07:05:00+00:00
8,29765,18,M,United States,Display,2026-08-27 07:46:00+00:00
9,31252,18,M,United States,Facebook,2024-09-18 08:30:00+00:00


## Salvando Tabelas no BigQuery

Para salvar uma tabela no BigQuery, precisamos de um job_config da classe bigquery.LoadJobConfig. Essa classe pode receber dois argumentos. Um write_disposition que determina o que acontece caso a tabela no local indicado já exista. Quando esse parâmetro recebe "WRITE_TRUNCATE", a tabela existente é substituída completamente. O segundo argumento que podemos utilizar é o autodetect. Quando recebe True, detecta automaticamente o tipo das colunas ao salvar a tabela.

Depois de definir o job_config, chamamos o método load_table_from_dataframe do bigquery_client. Esse método é o responsável por salvar a tabela. Como primeiro argumento ele deve receber um dataframe do pandas. Como segundo argumento, o método deve receber a localização da nova tabela. Essa localização deve estar no formato: ```<ID do projeto>.<Nome do dataset>.<Nome da tabela>```. Também é necessário, é claro, passa como argumento o job_config que vai configurar a requisição para salvar o dataframe. O retorno do método load_table_from_dataframe é um objeto com um método result, esse método vai esperar a tabela ser salva. Sem ele, o script do python encerra antes que a requisição termine.

In [8]:
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

job = bigquery_client.load_table_from_dataframe(dataframe, f"{PROJECT_ID}.meu_dataset.nova_tabela", job_config=job_config)
job.result()

print(f"Tabela salva com {len(dataframe)} linhas")

Tabela salva com 17554 linhas


## Utilizando BigFrames

Abaixo, temos um exemplo de como utilizar bigframes. Para termos acesso ao projeto do BigQuery, precisamos injetar o ID do projeto em bpd.options.bigquery.project. Dessa forma, quando chamarmos o método read_gbq do bpd, a biblioteca conseguirá acessar a tabela através da localicação dela. 

O bigframe suporta a maioria das funcionalidades do pandas e a maioria delas de fato manipula os dados como no pandas. Mas, algumas funcionalidades ainda não estão mapeadas e por isso podem não funcionar como o esperado. Nosso exemplo selecionou as colunas desejadas, filtrou os usuários por uma lista de países, filtrou pelo gênero e por idade. Logo depois, convertemos o bigframe para um dataframe do pandas contendo apenas os 100 primeiros registros. Nesse exato momento os dados estarão na memória local, porém, apenas 100 linhas de registros com as colunas que selecionamos.

In [9]:
import bigframes.pandas as bpd

bpd.options.bigquery.project = PROJECT_ID
bigframe = bpd.read_gbq("bigquery-public-data.thelook_ecommerce.users")

bigframe = bigframe[["id", "age", "country", "gender", "traffic_source", "created_at"]]
bigframe = bigframe[bigframe["country"].isin(["Brasil", "Japan", "United States"])]
bigframe = bigframe[bigframe["gender"] == "M"]
bigframe: bpd.DataFrame = bigframe[bigframe["age"] >= 18]

dataframe = bigframe.head(n=100).to_pandas()

Podemos também salvar normalmente o dataframe em formato parquet.

In [10]:
dataframe.to_parquet("../snapshots/users-with-bigframe.parquet")

E também podemos, é claro, salvar esse dataframe como uma tabela lá no BigQuery, assim como fizemos anteriormente.

In [11]:
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

job = bigquery_client.load_table_from_dataframe(dataframe, f"{PROJECT_ID}.meu_dataset.nova_tabela_com_bigframe", job_config=job_config)
job.result()

print(f"Tabela salva com {len(dataframe)} linhas")

Tabela salva com 100 linhas
